In [1]:
import torch
import faiss
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer, DPRQuestionEncoder, DPRQuestionEncoderTokenizer
import utils
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
import time

In [2]:
# 0. Debug CUDA
!nvidia-smi
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU compute capability: {torch.cuda.get_device_capability(0)}")

Wed Apr 22 13:35:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.10             Driver Version: 582.28         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1070        On  |   00000000:26:00.0  On |                  N/A |
| 24%   51C    P0             29W /  151W |     846MiB /   8192MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "facebook/dpr-ctx_encoder-single-nq-base"
tokenizer = DPRContextEncoderTokenizer.from_pretrained(model_name)
model = DPRContextEncoder.from_pretrained(model_name, use_safetensors=True).to(device)

# (for Retrieval step)
q_model_name = "facebook/dpr-question_encoder-single-nq-base"
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(q_model_name)
q_model = DPRQuestionEncoder.from_pretrained(q_model_name, use_safetensors=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# 1.1 Environment Variables

CORPUS_DIR = ".wiki"
PASSAGE_DB =".passage_meta.db"
INDEX_PATH = ".index.faiss"
QUESTIONS = "assets/questions.txt"

In [5]:
# 2. Get Wikipedia Passages
passages = []
passages_meta = []

print("Reading in chunked passages from corpus...")
for i, (path, title, passage) in enumerate(utils.yield_passages(CORPUS_DIR)):
    passages.append(passage)
    passages_meta.append((i,title,path))
print("Done.")

Reading in chunked passages from corpus...
Reading:  .wiki/enwiki-20140602-pages-articles.xml-1259.txt
Done.


In [6]:
# 2.1 Index Metadata (for Retrieval)

if os.path.exists(PASSAGE_DB): 
    print(f"DB exists: {PASSAGE_DB}. Skip indexing DB.")

else:
    utils.index_passages(PASSAGE_DB, passages_meta)

Indexing passages to .passage_meta.db...
Done.


In [ ]:
# 3. Batch Encoding

def encode_passages(passages, batch_size):
    all_embeddings = []
    model.eval()

    # We use tqdm here to wrap the range
    # total=len(passages)//batch_size tells it how many steps to expect
    pbar = tqdm(total=len(passages), desc="Encoding Passages", unit="psg")
    
    with torch.no_grad():
        for i in range(0, len(passages), batch_size):
            batch = passages[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            embeddings = model(**inputs).pooler_output.cpu().numpy()
            all_embeddings.append(embeddings)
            # Update progress bar by the size of the batch
            pbar.update(len(batch))
            
    pbar.close()
    print("Done encoding.")
    return np.vstack(all_embeddings)


if os.path.exists(INDEX_PATH): 
    print(f"Index exists: {INDEX_PATH}. Skip encoding step.")

else:
    print(f"Embeddings index not exists: {INDEX_PATH}. Encode passages.")
    embeddings = encode_passages(passages, batch_size=32) # on GTX 1070


Embeddings index not exists: .index.faiss. Encode passages.


Encoding Passages:   1%|▋                                                  | 17664/1398607 [03:22<4:14:04, 90.59psg/s]

In [ ]:
# 4. FAISS Index

if os.path.exists(INDEX_PATH): 
    print(f"Index exists: {INDEX_PATH}. Loading.")
    index = faiss.read_index(INDEX_PATH)

else:
    print(f"Indexing embeddings to {INDEX_PATH}")
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings) # from step 3
    faiss.write_index(index, INDEX_PATH)
    
    print(f"Indexed embeddings successfully: {INDEX_PATH}.")

In [ ]:
# 5. Evaluation

def search(query, k):
    '''Gets passage IDs'''
    inputs = q_tokenizer(query, return_tensors="pt")
    with torch.no_grad():
        q_embedding = q_model(**inputs).pooler_output.numpy()
    
    distances, indices = index.search(q_embedding, k)
    
    return [int(i) for i in indices[0]]
    

mrr = []

for category, clue, answers in utils.get_questions(QUESTIONS):

    # construct query: category + refined clue
    query = category + " " + utils.refine_query(clue)
    print(query)

    # retrieve ranked passage IDs
    K=100
    doc_ids = search(query, K)

    # get titles from passage IDs
    titles = [utils.fetch_row(PASSAGE_DB, pid)[0] for pid in doc_ids]
    print(titles[0],'\n')

    # compute Mean Reciprocal Rank
    for ans in answers:
        try:
            rank = 1.0 / (titles.index(ans) + 1.0)
        except ValueError:
            rank = 0.0
        if rank: break
            
    mrr.append(rank)

In [ ]:
# 5.1 Evaluation (Continued)
print(f"MRR@{K}", np.mean(mrr))